# M2.S3 — Distributed-memory computing with MPI
## Interactive HPC notebook

This notebook accompanies **M2.S3 — Distributed-memory computing with MPI**.

The key transition from the previous session is:

> **OpenMP shares memory. MPI moves data.**

We will make the main lecture concepts visible by running small MPI programs:

1. identify processes with **rank, size and communicator**;
2. see that every process owns **separate memory**;
3. exchange data with **send / receive / tags**;
4. reason about **message ordering and deadlock**;
5. use **broadcast, scatter, gather and reduce**;
6. observe that **communication is not free**;
7. connect **Slurm tasks** to MPI ranks across nodes.

### Classroom method
> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Recommended environment
- IE/SciTech JupyterHub
- Python 3 kernel
- MPI compiler/runtime (`mpicc`)
- Slurm

### Execution model
Sections **1–5** are small MPI teaching examples. They run directly from Jupyter with `mpirun`/`mpiexec` so students get immediate feedback and do not create a Slurm job for every tiny message-passing example.

Section **6** is an optional communication-cost illustration; its timings are qualitative only.

Section **7 uses Slurm**, because that is where multi-node allocation and rank placement actually matter.

So the rule is:

> **small MPI semantics demo → run directly**  
> **real multi-node allocation / placement → run with Slurm**

The notebook does **not** install MPI or system packages.

> **Notebook build: M2S3-2026-09-20-v3**
>
> If you previously opened an older M2.S3 notebook in this Jupyter session, use **Kernel → Restart Kernel and Run All Cells** before testing this version. Jupyter kernels keep old Python function definitions in memory even after a notebook file is replaced.
>
> In this build, Sections 1–6 use a helper named `run_mpi_demo()`. If you ever see **"Slurm launch: srun --ntasks ..."** in Sections 1–6, you are running an older notebook/kernel state.

# 0 — Environment check

### Predict
Before running the cell:

- Is `mpicc` available?
- Is `mpirun` or `mpiexec` available?
- Is Slurm visible from this Jupyter environment?
- Does seeing many CPUs mean that you automatically own them?

In [ ]:
import os
import platform
import shutil
import subprocess
import textwrap
import time

print('Host:', platform.node())
print('User:', os.environ.get('USER', 'unknown'))
print('Visible CPUs:', os.cpu_count())
print('mpicc:', shutil.which('mpicc'))
print('mpirun:', shutil.which('mpirun'))
print('mpiexec:', shutil.which('mpiexec'))
print('srun:', shutil.which('srun'))
print('sinfo:', shutil.which('sinfo'))

if shutil.which('mpicc'):
    p = subprocess.run(['mpicc', '--version'], capture_output=True, text=True)
    print('\nMPI compiler wrapper:')
    print((p.stdout or p.stderr).splitlines()[0] if (p.stdout or p.stderr) else 'available')
else:
    print('\nmpicc is not available. Run the MPI activities on the SciTech HPC environment.')
M2S3_NOTEBOOK_BUILD = "M2S3-2026-09-20-v4"
print("Notebook build:", M2S3_NOTEBOOK_BUILD)


In [ ]:
print("run_mpi_demo() helper loaded — small MPI examples will NOT use srun")
print("Teaching examples (Sections 1–6): direct MPI launch from Jupyter")
print("Real multi-node experiment (Section 7): Slurm batch job")

def compile_mpi(source_file, exe_file):
    if shutil.which('mpicc') is None:
        print('mpicc not found — run this notebook in the SciTech Jupyter/HPC environment.')
        return False
    p = subprocess.run(
        ['mpicc', '-O2', source_file, '-o', exe_file],
        capture_output=True, text=True
    )
    if p.returncode != 0:
        print('Compilation failed:')
        print(p.stderr)
        return False
    return True

def _mpi_launcher(n, exe_file):
    runner = shutil.which('mpirun') or shutil.which('mpiexec')
    if runner is None:
        return None, None

    version = subprocess.run(
        [runner, '--version'],
        capture_output=True, text=True
    )
    version_text = (version.stdout or '') + (version.stderr or '')

    # Open MPI enforces available "slots" by default. For these tiny classroom
    # examples we allow oversubscription so 2–4 toy ranks can run inside the
    # Jupyter environment even when the Jupyter session itself has a small CPU allocation.
    if 'Open MPI' in version_text or 'OpenRTE' in version_text:
        cmd = [runner, '--oversubscribe', '-np', str(n), './' + exe_file]
    else:
        cmd = [runner, '-n', str(n), './' + exe_file]

    return cmd, version_text

def run_mpi_demo(exe_file, n=4, timeout=60):
    """Run a small MPI teaching example directly from Jupyter."""
    cmd, version_text = _mpi_launcher(n, exe_file)
    if cmd is None:
        print('mpirun/mpiexec is not available. Run this notebook in the SciTech environment.')
        return None

    env = os.environ.copy()
    env.setdefault('OMPI_ALLOW_RUN_AS_ROOT', '1')
    env.setdefault('OMPI_ALLOW_RUN_AS_ROOT_CONFIRM', '1')

    print("Direct MPI launch:", " ".join(cmd))
    print("(Teaching example only — this is not a Slurm scaling run.)")

    try:
        p = subprocess.run(
            cmd, capture_output=True, text=True, env=env, timeout=timeout
        )
    except subprocess.TimeoutExpired:
        print('MPI run timed out.')
        return None

    if p.stdout:
        print(p.stdout, end='')
    if p.returncode != 0:
        print('\nMPI launch returned an error:')
        print(p.stderr)
        print('If the local MPI launcher is restricted in Jupyter, skip this cell and use the Slurm section later.')
    return p

def submit_slurm(script_path):
    if shutil.which('sbatch') is None:
        print('sbatch is not available here. Run Section 7 in the SciTech HPC environment.')
        return None
    submit_env = os.environ.copy()
    for key in ('SLURM_MEM_PER_CPU', 'SLURM_MEM_PER_GPU', 'SLURM_MEM_PER_NODE'):
        submit_env.pop(key, None)

    p = subprocess.run(
        ['sbatch', '--parsable', script_path],
        capture_output=True, text=True, env=submit_env
    )
    if p.returncode != 0:
        print('Slurm submission failed:')
        print(p.stderr)
        return None
    job_id = p.stdout.strip().split(';')[0]
    print('Submitted Slurm job:', job_id)
    return job_id

def slurm_status(job_id):
    if not job_id:
        print('No job id is available.')
        return
    if shutil.which('squeue'):
        p = subprocess.run(
            ['squeue', '-j', str(job_id), '-o', '%.18i %.9T %.10M %.6D %R'],
            capture_output=True, text=True
        )
        print(p.stdout if p.stdout.strip()
              else f'Job {job_id} is no longer in squeue (it may have completed).')
    else:
        print('squeue is not available.')

def show_job_output(job_id, prefix):
    if not job_id:
        print('No job id is available.')
        return
    path = f"{prefix}-{job_id}.out"
    if os.path.exists(path):
        print(open(path).read())
    else:
        print(f'{path} does not exist yet. Re-run this cell after the job finishes.')

### Explain
The MPI runtime launches **processes**. Slurm, when used, allocates resources to your job.

Those are related but different decisions.

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** MPI defines communicating **processes/ranks**. Slurm allocates the **hardware resources** used for real cluster jobs. In Sections 1–5 we use a few tiny local MPI ranks only to demonstrate semantics. In Section 7, Slurm performs the real multi-node allocation and placement.

</details>


# 1 — Rank, size and communicator
### Slide connection: every MPI process needs an identity

Every process executes the same program, but each process can discover:

> For this tiny identity example we use a direct local MPI launch. We are learning **rank/size semantics**, not measuring cluster performance.

- its **rank**: identity inside a communicator;
- the **size**: number of processes in that communicator;
- the processor/host where it is running.

### Predict
If we launch 4 processes:

1. What ranks should exist?
2. Will rank 3 have higher priority than rank 1?
3. Will the printed lines necessarily appear in rank order?

In [ ]:
hello_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank, size, len;
    char name[MPI_MAX_PROCESSOR_NAME];
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Get_processor_name(name, &len);

    printf("Hello from rank %d of %d on %s\n", rank, size, name);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_hello.c', 'w') as f:
    f.write(hello_src)

if compile_mpi('mpi_hello.c', 'mpi_hello'):
    run_mpi_demo('mpi_hello', 4)

### Observe
- Ranks should be `0 ... size-1`.
- Rank is an **identifier**, not a resource priority.
- Output ordering can vary because processes execute asynchronously.

### Try it
Change the launch from 4 processes to 2, then 6 (if your environment permits it).

### Explain
Why can the same executable behave differently on each process even though every process starts from the same source code?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** every rank starts the same program, but `MPI_Comm_rank()` gives each process a different rank. Code can branch on that rank and each process has its own local state, so the same executable can perform different roles.

</details>


# 2 — Separate memory: changing Rank 0 does not change Rank 1
### Slide connection: MPI does not make remote memory shared

Each process has its own local memory.

### Predict
Each rank starts with `value = rank * 10`.

Rank 0 then changes **its own** value to 999.

Will any other rank automatically see 999?

In [ ]:
memory_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int value = rank * 10;
    printf("Rank %d initially has value=%d\n", rank, value);

    MPI_Barrier(MPI_COMM_WORLD);
    if (rank == 0) value = 999;
    MPI_Barrier(MPI_COMM_WORLD);

    printf("Rank %d after Rank 0 changes its copy: value=%d\n", rank, value);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_separate_memory.c', 'w') as f:
    f.write(memory_src)

if compile_mpi('mpi_separate_memory.c', 'mpi_separate_memory'):
    run_mpi_demo('mpi_separate_memory', 4)

### Observe
Only Rank 0 changes to 999.

### Explain
How is this fundamentally different from OpenMP threads reading and writing one shared variable?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** MPI processes normally have separate address spaces. Changing Rank 0's local variable does not modify Rank 1's copy. OpenMP threads inside one process share the same address space, so shared variables can be directly read/written by multiple threads.

</details>


# 3 — Point-to-point communication: send, receive and tag
### Slide connection: one sender · one receiver · one matching communication

We model a weather-boundary update:

- Rank 0 owns a local boundary temperature;
- Rank 1 cannot read Rank 0's variable directly;
- Rank 0 sends a **copy**;
- Rank 1 receives it using matching source/tag/communicator information.

### Predict
What should Rank 1 know after the receive?

Can Rank 1 discover which rank sent the message and which tag was used?

In [ ]:
p2p_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    const int tag = 42;
    if (rank == 0) {
        double boundary_temp = 18.4;
        printf("Rank 0 local value before send = %.1f C\n", boundary_temp);
        MPI_Send(&boundary_temp, 1, MPI_DOUBLE, 1, tag, MPI_COMM_WORLD);
    } else {
        double received_temp = -999.0;
        MPI_Status status;
        MPI_Recv(&received_temp, 1, MPI_DOUBLE, 0, tag, MPI_COMM_WORLD, &status);
        printf("Rank 1 received %.1f C from rank %d with tag %d\n",
               received_temp, status.MPI_SOURCE, status.MPI_TAG);
    }

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_point_to_point.c', 'w') as f:
    f.write(p2p_src)

if compile_mpi('mpi_point_to_point.c', 'mpi_point_to_point'):
    run_mpi_demo('mpi_point_to_point', 2)

### Observe
The receiver obtains a copy through explicit communication.

### Explain
A message match depends on **source/destination + tag + communicator**. Why is the tag useful when the same pair of ranks exchanges several different kinds of data?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** tags label the **kind or phase of a message**. The same two ranks may exchange temperature, pressure and control messages; different tags let the receiver match the intended message rather than merely “anything from Rank 0”.

</details>


# 4 — Message ordering and deadlock
### Slide connection: correct data · wrong communication order

Consider this unsafe pattern with two ranks:

```c
// Rank 0
MPI_Recv(... from Rank 1 ...);
MPI_Send(... to Rank 1 ...);

// Rank 1
MPI_Recv(... from Rank 0 ...);
MPI_Send(... to Rank 0 ...);
```

### Predict
What event can make progress first?

**Do not execute a deliberately hanging program in a shared class notebook.**

Instead, run the repaired version below: one side sends first, the other receives first.

In [ ]:
ordered_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    int send_value = (rank == 0) ? 100 : 200;
    int recv_value = -1;

    if (rank == 0) {
        MPI_Send(&send_value, 1, MPI_INT, 1, 7, MPI_COMM_WORLD);
        MPI_Recv(&recv_value, 1, MPI_INT, 1, 8, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    } else {
        MPI_Recv(&recv_value, 1, MPI_INT, 0, 7, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(&send_value, 1, MPI_INT, 0, 8, MPI_COMM_WORLD);
    }

    printf("Rank %d received %d\n", rank, recv_value);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_ordered_exchange.c', 'w') as f:
    f.write(ordered_src)

if compile_mpi('mpi_ordered_exchange.c', 'mpi_ordered_exchange'):
    run_mpi_demo('mpi_ordered_exchange', 2)

### Observe
The first matching send/receive can complete, so the second exchange can then proceed.

### Explain
- Why did the original receive-first/receive-first pattern have no possible first step?
- How could non-blocking communication (`MPI_Isend`, `MPI_Irecv`, `MPI_Wait`) provide another solution?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** both ranks enter a blocking receive before either rank has sent anything, so neither process can make progress. Non-blocking calls can post sends/receives first and then use `MPI_Wait` to complete them, avoiding that circular wait when used correctly.

</details>


# 5 — Collectives: match the operation to the communication pattern
### Slide connection: broadcast · scatter · gather · reduce

We will execute four common patterns in one program:

1. **Broadcast** — Rank 0 shares a timestep with everyone.
2. **Scatter** — Rank 0 divides an 8-element image among 4 ranks.
3. **Gather** — processed pieces return to Rank 0.
4. **Reduce** — local energies become one total energy.

### Predict
With 4 processes:

- What timestep should every rank print?
- Which two image values should each rank receive?
- What should the gathered processed image contain?
- If local energy is `rank + 1`, what should the reduced total be?

In [ ]:
collective_src = r'''
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 4) {
        if (rank == 0) printf("Run this example with exactly 4 processes.\n");
        MPI_Finalize();
        return 0;
    }

    // 1) Broadcast
    int timestep = (rank == 0) ? 60 : -1;
    MPI_Bcast(&timestep, 1, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Rank %d: timestep=%d\n", rank, timestep);

    MPI_Barrier(MPI_COMM_WORLD);

    // 2) Scatter
    int image[8] = {1,2,3,4,5,6,7,8};
    int chunk[2] = {-1,-1};
    MPI_Scatter(image, 2, MPI_INT, chunk, 2, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Rank %d received image chunk [%d,%d]\n", rank, chunk[0], chunk[1]);

    // Local processing
    chunk[0] *= 10;
    chunk[1] *= 10;

    // 3) Gather
    int processed[8] = {0};
    MPI_Gather(chunk, 2, MPI_INT, processed, 2, MPI_INT, 0, MPI_COMM_WORLD);
    if (rank == 0) {
        printf("Gathered processed image:");
        for (int i = 0; i < 8; ++i) printf(" %d", processed[i]);
        printf("\n");
    }

    // 4) Reduce
    int local_energy = rank + 1;
    int total_energy = 0;
    MPI_Reduce(&local_energy, &total_energy, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
    if (rank == 0)
        printf("Reduced total energy = %d\n", total_energy);

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_collectives.c', 'w') as f:
    f.write(collective_src)

if compile_mpi('mpi_collectives.c', 'mpi_collectives'):
    run_mpi_demo('mpi_collectives', 4)

### Observe

**Expected values with 4 ranks**
- timestep on every rank: **60**
- image chunks: Rank 0 `[1,2]`, Rank 1 `[3,4]`, Rank 2 `[5,6]`, Rank 3 `[7,8]`
- gathered processed image: **10 20 30 40 50 60 70 80**
- reduced total energy: **10**

- **Broadcast:** one value → everyone.
- **Scatter:** different data pieces → different ranks.
- **Gather:** different pieces → one root.
- **Reduce:** local values → one combined result.

### Explain
Could you reproduce all of these patterns using many individual sends and receives?

Why is using the collective that matches the problem usually clearer?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** yes, you could manually implement these patterns with many point-to-point messages, but the code becomes longer and easier to get wrong. MPI collectives state the communication pattern directly and MPI implementations can optimize that pattern for the system.

</details>


# 6 — Communication is not free
### Slide connection: moving data and waiting can limit speedup

This optional micro-experiment measures a simple ping-pong between two local MPI ranks.

**Important:** this is only a qualitative teaching illustration. The ranks may share the same Jupyter host and may be oversubscribed, so the numbers are **not** a cluster-network benchmark.

Its purpose is simply to make one point visible:

> Communication takes time, and message size affects that cost.

### Predict
Which message size should have the smallest absolute round-trip time?

Why might larger messages achieve higher effective bandwidth even though each round trip takes longer?

In [ ]:
ping_src = r'''
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

static void test_size(int count, int reps, int rank) {
    double *buf = (double*)malloc((size_t)count * sizeof(double));
    for (int i = 0; i < count; ++i) buf[i] = (double)i;

    MPI_Barrier(MPI_COMM_WORLD);
    double t0 = MPI_Wtime();

    for (int r = 0; r < reps; ++r) {
        if (rank == 0) {
            MPI_Send(buf, count, MPI_DOUBLE, 1, 1, MPI_COMM_WORLD);
            MPI_Recv(buf, count, MPI_DOUBLE, 1, 2, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        } else {
            MPI_Recv(buf, count, MPI_DOUBLE, 0, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            MPI_Send(buf, count, MPI_DOUBLE, 0, 2, MPI_COMM_WORLD);
        }
    }

    double elapsed = MPI_Wtime() - t0;
    if (rank == 0) {
        double bytes = (double)count * sizeof(double);
        double avg_rtt_us = elapsed * 1e6 / reps;
        double mb_s = (2.0 * bytes * reps) / elapsed / 1e6;
        printf("%9.1f KiB : avg round trip = %9.2f us, effective transfer = %9.1f MB/s\n",
               bytes / 1024.0, avg_rtt_us, mb_s);
    }
    free(buf);
}

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (size != 2) {
        if (rank == 0) printf("Run this example with exactly 2 processes.\n");
        MPI_Finalize();
        return 0;
    }

    test_size(1,       200, rank);      // 8 bytes
    test_size(1024,    100, rank);      // 8 KiB
    test_size(131072,   20, rank);      // 1 MiB

    MPI_Finalize();
    return 0;
}
'''

with open('mpi_pingpong_cost.c', 'w') as f:
    f.write(ping_src)

if compile_mpi('mpi_pingpong_cost.c', 'mpi_pingpong_cost'):
    run_mpi_demo('mpi_pingpong_cost', 2, timeout=45)

### Observe
Interpret the result **qualitatively only**. Do not compare these numbers between students or treat them as SciTech network performance.

### Explain
If a simulation keeps splitting the problem into smaller regions, why can communication and waiting eventually dominate useful computation?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** increasing the number of regions reduces useful local computation per rank, while boundary exchanges, message startup and waiting do not necessarily shrink at the same rate. Eventually communication/synchronization can consume a large fraction of the timestep.

</details>


# 7 — Real cluster run: Slurm allocation → MPI ranks
### Slide connection: scheduler allocation and MPI process identity

Now we move from small local teaching examples to the **real HPC cluster**.

We request:

```text
2 nodes
4 MPI tasks
2 tasks per node
```

This is where Slurm is necessary: it allocates the nodes and places the MPI ranks.

The cells below create the batch script, submit it directly from Jupyter, check the queue and display the result.

### Predict
If Slurm grants the request, how many rank IDs should the program print?

Could two ranks share the same hostname?

Should ranks 0–3 necessarily be assigned to specific physical CPU cores?

In [ ]:
slurm_script = '''#!/bin/bash
#SBATCH --job-name=m2s3_mpi
#SBATCH --partition=cpu
#SBATCH --nodes=2
#SBATCH --ntasks=4
#SBATCH --ntasks-per-node=2
#SBATCH --time=00:03:00
#SBATCH --output=m2s3_mpi-%j.out

set -e

# JupyterHub itself is launched through Slurm on SciTech. Remove inherited
# memory-policy variables before creating a new job step.
unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== Allocation ==="
echo "SLURM_JOB_ID=$SLURM_JOB_ID"
echo "SLURM_JOB_NUM_NODES=$SLURM_JOB_NUM_NODES"
echo "SLURM_NTASKS=$SLURM_NTASKS"
echo

echo "Allocated nodes:"
scontrol show hostnames "$SLURM_JOB_NODELIST"
echo

echo "=== MPI ranks ==="
mpicc -O2 mpi_hello.c -o mpi_hello
srun ./mpi_hello
'''

with open('m2s3_mpi.slurm', 'w') as f:
    f.write(slurm_script)

print(slurm_script)

In [ ]:
if shutil.which('sinfo') is None:
    print('Slurm is not visible here. Run Section 7 in the SciTech HPC environment.')
else:
    print('--- SciTech Slurm partitions ---')
    subprocess.run(['sinfo', '-o', '%P %a %l %D %c'], check=False)
    print("\nThe course cluster guide documents 'cpu' as the CPU batch partition.")

### Submit the two-node MPI job directly from Jupyter

Run this cell once. Slurm will queue the request if two nodes are not immediately available.

In [ ]:
M2S3_JOB_ID = submit_slurm('m2s3_mpi.slurm')

### Check the job

Re-run this cell while the job is pending/running.

In [ ]:
slurm_status(globals().get('M2S3_JOB_ID'))

### Read the result

Run after the job completes. The output should show four ranks and the hostnames selected by Slurm.

In [ ]:
show_job_output(globals().get('M2S3_JOB_ID'), 'm2s3_mpi')

### Observe

After the job completes, look for:

- **2 allocated node names**;
- **4 rank IDs**;
- two ranks sharing each node hostname if the requested layout is honored.

### Explain

- **Slurm** allocates and places resources.
- **MPI** gives each launched process a rank and coordinates communication.
- Why is an MPI rank not the same thing as a physical CPU core or node?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** a rank is a logical identifier inside an MPI communicator. Slurm decides the physical placement. In this job, four ranks are distributed over two allocated nodes, with two tasks requested per node. The rank number itself does not mean “CPU core number” or “node number”.

</details>

# Challenge — Distributed image processing

Work in pairs.

You have a grayscale image represented as 16 integer pixels on Rank 0:

```text
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
```

Your task is to design an MPI solution for 4 ranks:

1. Rank 0 owns the full image initially.
2. Split it evenly across 4 ranks.
3. Each rank applies `pixel = 255 - pixel` to its local chunk.
4. Reconstruct the final image on Rank 0.
5. Each rank computes the sum of its processed pixels.
6. Rank 0 obtains the global sum.

### Before coding, choose the MPI operation for each step
- distribute image chunks → **?**
- reconstruct image → **?**
- combine local sums → **?**

Use the collective example above as your starting point rather than typing a large program from scratch.

<details>
<summary><strong>Show suggested solution</strong></summary>

Use the collective that matches each data movement:

- distribute 16 pixels from Rank 0 → **`MPI_Scatter`**
- reconstruct the processed image on Rank 0 → **`MPI_Gather`**
- combine one local sum from each rank → **`MPI_Reduce(..., MPI_SUM, ...)`**

With four ranks, each receives four pixels:

- Rank 0: `0 1 2 3` → `255 254 253 252`
- Rank 1: `4 5 6 7` → `251 250 249 248`
- Rank 2: `8 9 10 11` → `247 246 245 244`
- Rank 3: `12 13 14 15` → `243 242 241 240`

The reconstructed image is `255 ... 240` and the global sum is **3960**.

</details>


# What did we learn?

1. **MPI processes own separate memory.** Remote variables are not directly shared.
2. **Rank identifies a process inside a communicator.** Rank is not a priority.
3. **Point-to-point messages must match.** Source/destination, tag and communicator matter.
4. **Communication ordering matters.** A valid calculation can still hang if no process can make progress.
5. **Collectives express common group patterns:** broadcast, scatter, gather and reduce.
6. **Communication has a cost.** Good MPI designs do enough useful local work between communications.
7. **Slurm allocates resources; MPI processes run on those resources.**

## Next: M2.S4 — Accelerator computing

Distributed CPU processes are only one level of modern HPC. In the next session we ask:

> If every node also has a GPU, which work should stay on the CPU and which work should run on the accelerator?